<a href="https://colab.research.google.com/github/jadenvv/selfAttentionSentiment/blob/main/selfAttentionSentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [180]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
from google.colab import userdata
os.environ['KAGGLE_KEY']= userdata.get("Kaggle_key")
os.environ['KAGGLE_USERNAME']= userdata.get("Kaggle_user")
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import Embedding
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:

training_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,"jp797498e/twitter-entity-sentiment-analysis", "twitter_training.csv")
training_df.columns = ["ID","TOPIC",'SENTIMENT', "TWEET"]
training_df.drop_duplicates(subset="TWEET", inplace=True,  ignore_index=True)
training_df.dropna(subset=["TWEET", "SENTIMENT"], inplace=True, ignore_index=True)
sent,reverse_indices  = np.unique(training_df["SENTIMENT"], return_inverse=True)
training_df.SENTIMENT = reverse_indices
training_df['TWEET'] = [x+ " <EOS>"for x in training_df["TWEET"]]
vocab = list(set((" ".join(training_df["TWEET"].tolist())).split(' ')))
stoi = {text:(idx+1) for idx,text in enumerate(vocab)}
itos = {(1+idx):text for idx,text in enumerate(vocab)}
stoi["<PAD>"] = 0
itos[0] = "<PAD>"


Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.


## Getting the context size
decided on a context size of 64 since the way I create the vocab is based on splitting on whitespace so granted we are gonna get

Addionally,  99 percent of the dataset has lenghts less than 64

In [ ]:
n = np.array([len(x.split(' ')) for x in training_df.TWEET])
over = training_df.loc[n > 64 ]
len(over.TWEET.iloc[1].split(' '))


66

In [ ]:
np.percentile(n, 99)

np.float64(60.0)

In [ ]:
class SentDataset(Dataset):
  def __init__(self,df, context=64):
      self.context = context
      self.sentiment = df["SENTIMENT"]
      self.tweet_data= self.homogenize([
          np.array([stoi[x] for x in tweet.split(' ')])
          for tweet in df.TWEET
      ])

  def homogenize(self,arr):
    return np.array([
        sentence[:self.context]
        if len(sentence) >self.context
        else np.pad(sentence, (0,self.context -len(sentence)))
        for sentence in arr

    ])
  def __getitem__(self,idx):
      tweet = self.tweet_data[idx]
      sentiment = self.sentiment.iloc[idx]
      return (tweet,sentiment)
  def __len__(self):
      return len(self.tweet_data)

In [ ]:
sentdata  = SentDataset(training_df)
dataLoaderSent=DataLoader(sentdata,64, shuffle=True, num_workers=2,drop_last=True)

In [218]:
class attentionEmbedding(nn.Module):
  def __init__(self,embedding_dim:int,context:int):
    super().__init__()
    self.word_embedding =Embedding(len(vocab),embedding_dim,padding_idx=0)
    self.positional_embedding = self.getPositionalEmbedding(embedding_dim,context)
  def getPositionalEmbedding(self, embedd_dim, context):
    ret = torch.zeros((context,embedd_dim ), requires_grad=False)
    for k in range(context ):
      for x in range(embedd_dim):


  def forward(self,input):
    return self.word_embedding(input)

In [219]:
embedding = attentionEmbedding(128,32)
token,_ =next(iter(dataLoaderSent))

torch.Size([128, 32])


64